In [ ]:
# ---------------- Imports ----------------
import json
import os
from collections import defaultdict

import pandas as pd
import yaml
from sklearn.metrics import confusion_matrix, recall_score
import numpy as np



In [ ]:
# ---------------- Args ----------------
RESULTS_FILENAME = "20260210t165755-20260210T1651-llama-3.2-3b-instruct-20260115T095923-combined-claims-4k-authoritative-1.0"



In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
RESULTS_FOLDER = os.path.join(PROJ_STORE, "experiments", "model-evaluate")

# Results
RESULTS_PATH = os.path.join(RESULTS_FOLDER, f"{RESULTS_FILENAME}.jsonl")

# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "accuracy")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{RESULTS_FILENAME}.csv")
                          




In [ ]:
# -------------------------
# Load results
# -------------------------


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


In [ ]:

# -------------------------
# Metrics
# -------------------------

def compute_metrics_df(rows):
    df = pd.DataFrame(rows)

    POS = "SUPPORTS"
    NEG = "REFUTES"

    # Basic indicators
    df["correct"] = df["predicted_label"] == df["true_label"]
    df["pred_supports"] = df["predicted_label"] == POS
    df["true_supports"] = df["true_label"] == POS

    # Probability assigned to true label
    def true_label_prob(row):
        return row["scores"][row["true_label"]]["word_cond_prob"]

    # Probability assigned to SUPPORTS
    def supports_prob(row):
        return row["scores"][POS]["word_cond_prob"]

    df["true_label_prob"] = df.apply(true_label_prob, axis=1)
    df["supports_prob"] = df.apply(supports_prob, axis=1)

    rows_out = []

    def summarize(group_df, framing):
        y_true = group_df["true_label"]
        y_pred = group_df["predicted_label"]

        # Confusion matrix with fixed label order
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=[POS, NEG],
        )

        # Handle degenerate cases safely
        if cm.shape == (2, 2):
            TP, FN = cm[0]
            FP, TN = cm[1]
        else:
            TP = FN = FP = TN = 0

        n = len(group_df)

        accuracy = (TP + TN) / n if n else np.nan
        tpr = TP / (TP + FN) if (TP + FN) else np.nan
        tnr = TN / (TN + FP) if (TN + FP) else np.nan
        fpr = FP / (FP + TN) if (FP + TN) else np.nan
        fnr = FN / (FN + TP) if (FN + TP) else np.nan
        balanced_acc = np.nanmean([tpr, tnr])

        pct_supports = group_df["pred_supports"].mean()

        mean_true_prob = group_df["true_label_prob"].mean()

        # Confidence on misinformation
        refutes_df = group_df[group_df["true_label"] == NEG]
        mean_supports_prob_on_refutes = (
            refutes_df["supports_prob"].mean()
            if len(refutes_df) > 0 else np.nan
        )

        return {
            "framing_type": framing,
            "n": n,
            "accuracy": accuracy,
            "balanced_accuracy": balanced_acc,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "TPR_recall": tpr,
            "TNR_specificity": tnr,
            "FPR": fpr,
            "FNR": fnr,
            "pct_pred_supports": pct_supports,
            "prob_weighted_correctness": mean_true_prob,
            "mean_supports_prob_on_refutes": mean_supports_prob_on_refutes,
        }

    # Per framing_type
    for framing, g in df.groupby("framing_type"):
        rows_out.append(summarize(g, framing))

    # Overall
    rows_out.append(summarize(df, "OVERALL"))

    return pd.DataFrame(rows_out)


In [ ]:
# -------------------------
# Usage
# -------------------------

rows = load_jsonl(RESULTS_PATH)

metrics_df = compute_metrics_df(rows)

metrics_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nWrote metrics to {OUTPUT_FILE}")
display(metrics_df)
